In [5]:

import pandas as pd
import os, json, re
import fitz
from pathlib import Path
from dataclasses import asdict
from utils import Helper
utils = Helper()


In [6]:
from dataclasses import dataclass, asdict
import re

@dataclass
class PageGeometry:
    width: float
    height: float
    rotation: int
    orientation: str

@dataclass
class PageFeatures:
    char_count: int = 0
    non_space_chars: int = 0
    word_count: int = 0
    alpha_chars: int = 0
    digit_chars: int = 0
    number_count: int = 0
    numeric_chars: int = 0
    digit_ratio: float = 0.0
    numeric_ratio: float = 0.0
    word_to_number_ratio: float = 0.0
    text_density: float = 0.0

@dataclass
class PageSummary:
    page_number: int
    type: str                
    geometry: PageGeometry
    language: str | None    
    features: PageFeatures

# --- Pipeline functions ---

def detect_geometry(page):
    w, h = page.rect.width, page.rect.height
    orientation = "landscape" if w > h else "portrait"
    return PageGeometry(width=w, height=h, rotation=page.rotation, orientation=orientation)

def classify_language(text, threshold=0.8)->str:
    words = re.findall(r"\w+", text)
    english_count = sum(1 for w in words if re.match(r"^[a-zA-Z]+$", w))
    ratio = english_count / max(len(words), 1)
    return "English" if ratio >= threshold else "Non-English"

def page_text_or_scanned(page)->str:

    text = page.get_text("text").strip()
    page_rect = page.rect
    w = page.rect.width
    h = page.rect.height
    page_area = w * h

    if page_area <= 0:
        return "scanned"
    
    image_area = 0
    for img in page.get_images(full=True):
        try:
            xref = img[0]
            for rect in page.get_image_rects(xref):
                clipped = rect & page_rect
                if clipped.is_empty:
                    continue

                rect_area = clipped.width * clipped.height
                # Ignore small logos/icons
                if rect_area > page_area * 0.05:
                    image_area += rect_area

        except Exception:
            continue

    image_coverage = min(image_area / page_area, 1.0)
    blocks = page.get_text("blocks")
    text_blocks = [
        block for block in blocks if len(block) >= 5 and str(block[4]).strip()
    ]

    num_text_blocks = len(text_blocks)

    # Strong text page
    if len(text) > 100 and num_text_blocks >= 3 and image_coverage < 0.8:
        return "text"
    # Strong scanned page
    if image_coverage > 0.8 and len(text) < 100:
        return "scanned"
    # OCR scanned page
    if image_coverage > 0.9 and num_text_blocks <= 2:
        return "scanned"
    return "text" if len(text) > 100 else "scanned"

def page_features(page)->dict:

    text = page.get_text("text")
    text = re.sub(r"\s+", " ", text).strip()
    #char
    char_count = len(text)
    non_space_chars = len(text.replace(" ", ""))

    # alpha / digit counts
    alpha_chars = len(re.findall(r"[A-Za-z]", text))
    digit_chars = len(re.findall(r"\d", text))

    # words
    words = re.findall(r"[A-Za-z]+", text)
    word_count = len(words)

    # numeric values
    numbers = re.findall(
        r'\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b|\b\d+\.\d+\b|\b\d+\b',
        text
    )

    number_count = len(numbers)

    # numeric characters excluding commas and decimals
    numeric_chars = sum(
        len(re.sub(r"[,.]", "", x))
        for x in numbers
    )

    # ratios
    digit_ratio = (
        digit_chars / max(alpha_chars + digit_chars, 1)
    )

    numeric_ratio = (
        numeric_chars / max(non_space_chars, 1)
    )

    word_to_number_ratio = (
        word_count / max(number_count, 1)
    )

    # page area density
    page_area = page.rect.width * page.rect.height

    text_density = (
        non_space_chars / max(page_area, 1)
    )

    return {
        "char_count": char_count,
        "non_space_chars": non_space_chars,
        "word_count": word_count,
        "alpha_chars": alpha_chars,
        "digit_chars": digit_chars,
        "number_count": number_count,
        "numeric_chars": numeric_chars,
        "digit_ratio": round(digit_ratio, 4),
        "numeric_ratio": round(numeric_ratio, 4),
        "word_to_number_ratio": round(word_to_number_ratio, 2),
        "text_density": round(text_density, 6),
    }

def build_page_summary(page, page_number):
    geometry = detect_geometry(page)
    page_type = page_text_or_scanned(page)

    if page_type == "scanned":
        return PageSummary(page_number, "scanned", geometry, None, PageFeatures())

    # Text page → compute features + language
    feats = page_features(page)
    lang = classify_language(page.get_text())
    return PageSummary(page_number, "text", geometry, lang, PageFeatures(**feats))




In [ ]:
folder_pdf = r"C:\Users\kaustubh.keny\Projects\INPUTS\TERM SHEETS SMALL"

results = []

for idx,file in enumerate(os.listdir(folder_pdf)):

    file_path = os.path.join(folder_pdf, file)
    stem = Path(file).name
    print(f"{idx}: {stem}")
    doc = fitz.open(file_path)


    for idx in range(0, doc.page_count):

        summary = build_page_summary(doc[idx], idx + 1)
        d = asdict(summary)
        # Flatten nested dicts
        out = {
            "pdf_name": stem,
            "page_number": d["page_number"],
            "type": d["type"],
            "language": d["language"],
            **d["geometry"],
            **d["features"]
        }

        results.append(out)

    doc.close()



In [8]:
summary_df = pd.DataFrame(results)
summary_df.to_excel("page_summary.xlsx", index=False)

In [15]:
path =r"PDF_PAGE.xlsx"
folder_pdf = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\RANDOM_FETCH_DATA\NSE_FETCH\ANNUAL_REPORTS_2026"

df = pd.read_excel(path, sheet_name="2026")
df.head(4)

,pdf_name,page_number,page_type
0,2026_New_AADHARHFC.pdf,1,text
1,2026_New_AADHARHFC.pdf,2,text
2,2026_New_AADHARHFC.pdf,3,text
3,2026_New_AADHARHFC.pdf,4,text


In [ ]:

folder_pdf = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\RANDOM_FETCH_DATA\NSE_FETCH\ANNUAL_REPORTS_2026"
for file in os.listdir(folder_pdf):

    file_path = os.path.join(folder_pdf, file)

    stem = Path(file_path).name
    print(f"\nProcessing: {stem}")

    doc = fitz.open(file_path)

    mask = df["pdf_name"].str.contains(stem, na=False)

    # print("Matched row:", mask.sum())

    for idx, row in df.loc[mask].iterrows():

        # print("Row:", idx)

        page_n = int(row["page_number"]) - 1

        # res = page_text_or_scanned(doc[page_n])
        res = page_features(doc[page_n])
        # res = detect_orient(doc[page_n])
        
        for key, value in res.items():
            df.at[idx, key] = value           

    doc.close()

print(df.head())


#5m 24s 60k pages

In [ ]:
# Overwrite original file
df.to_excel(path, sheet_name="2026",index=False)

print(f"Saved updates to: {path}")